# Next-word prediction with an LSTM
 Tokenize FAQ text, build n-gram sequences, train a 2-layer LSTM, generate text.

In [8]:
import requests

url = "https://raw.githubusercontent.com/madhavkhurana1005/Next-Word-Prediction-Using-LSTM-GRU/main/hamlet.txt"

response = requests.get(url)
response.raise_for_status()

with open("hamlet.txt", "w", encoding="utf-8") as f:
    f.write(response.text)

print(response.text[:2000])

[The Tragedie of Hamlet by William Shakespeare 1599]


Actus Primus. Scoena Prima.

Enter Barnardo and Francisco two Centinels.

  Barnardo. Who's there?
  Fran. Nay answer me: Stand & vnfold
your selfe

   Bar. Long liue the King

   Fran. Barnardo?
  Bar. He

   Fran. You come most carefully vpon your houre

   Bar. 'Tis now strook twelue, get thee to bed Francisco

   Fran. For this releefe much thankes: 'Tis bitter cold,
And I am sicke at heart

   Barn. Haue you had quiet Guard?
  Fran. Not a Mouse stirring

   Barn. Well, goodnight. If you do meet Horatio and
Marcellus, the Riuals of my Watch, bid them make hast.
Enter Horatio and Marcellus.

  Fran. I thinke I heare them. Stand: who's there?
  Hor. Friends to this ground

   Mar. And Leige-men to the Dane

   Fran. Giue you good night

   Mar. O farwel honest Soldier, who hath relieu'd you?
  Fra. Barnardo ha's my place: giue you goodnight.

Exit Fran.

  Mar. Holla Barnardo

   Bar. Say, what is Horatio there?
  Hor. A peece of

In [10]:
data = response.text[:5000]

In [3]:
import re
import time
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## Tokenizer
Replaces `keras.preprocessing.text.Tokenizer` (same behaviour: lowercase, strip punctuation, index 1..N by frequency; 0 is reserved for padding).

In [11]:
# SimpleTokenizer class prepares raw text data for natural language processing (NLP) models by
# mapping words to unique numeric integers.
class SimpleTokenizer:
    FILTERS = '!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'

    def __init__(self):
        self.word_index = {} # dictionary that will map each unique word to a unique integer index
        self.index_word = {} # reverse dictionary of word_index, mapping integer indices back to their corresponding words
        self._table = str.maketrans({c: " " for c in self.FILTERS}) # maps all punctuation and control characters specified in FILTERS to a space (" ")

    def _split(self, text):
      # It converts the input text to lowercase using .lower().
      # It removes all punctuation by applying the translation table with .translate(self._table).
      # It splits the cleaned text into a list of individual words using .split().
        return text.lower().translate(self._table).split()

    def fit_on_texts(self, texts): # this method builds the vocabulary from a list of documents or strings (texts).
        counts = Counter() #counts the occurrences of each word across all documents using a Counter.
        for t in texts:
            counts.update(self._split(t))
        # then populates self.word_index and self.index_word by sorting words based on their frequency
        self.word_index = {w: i + 1 for i, (w, _) in enumerate(counts.most_common())}
        self.index_word = {i: w for w, i in self.word_index.items()}

    def texts_to_sequences(self, texts): # This method converts the raw strings in texts into lists of integers (sequences).
        # It cleans and splits each text, checks if each word exists in self.word_index, and replaces known words with their corresponding integer index.
        return [[self.word_index[w] for w in self._split(t) if w in self.word_index] for t in texts]



In [12]:
tokenizer = SimpleTokenizer()
tokenizer.fit_on_texts([data])
print(tokenizer)

In [13]:
vocab_size = len(tokenizer.word_index) + 1   # +1 for the padding index 0
print(vocab_size)

395


In [14]:
tokenizer = SimpleTokenizer()
tokenizer.fit_on_texts([data])
vocab_size = len(tokenizer.word_index) + 1   # +1 for the padding index 0
print("Unique words:", len(tokenizer.word_index), "| vocab_size:", vocab_size)

Unique words: 394 | vocab_size: 395


## Build n-gram input sequences

In [15]:
input_sequences = []
for sentence in data.split("\n"):
    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]
    for i in range(1, len(tokenized_sentence)):
        input_sequences.append(tokenized_sentence[: i + 1])

max_len = max(len(x) for x in input_sequences)
print("Num sequences:", len(input_sequences), "| max_len:", max_len)

Num sequences: 754 | max_len: 10


## Pad (pre-padding), split into X / y

In [16]:
def pad_sequences(seqs, maxlen):
    """Pre-pad with 0 (and pre-truncate if longer), like Keras pad_sequences."""
    out = np.zeros((len(seqs), maxlen), dtype=np.int64)
    for i, s in enumerate(seqs):
        s = s[-maxlen:]
        out[i, maxlen - len(s):] = s
    return out


padded_input_sequences = pad_sequences(input_sequences, max_len)

X = padded_input_sequences[:, :-1]
y = padded_input_sequences[:, -1]     # integer class labels; no one-hot needed in PyTorch
print(X.shape, y.shape)

(754, 9) (754,)


In [17]:
X_t = torch.from_numpy(X).long()
y_t = torch.from_numpy(y).long()

dataset = TensorDataset(X_t, y_t)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

## Model
Two stacked LSTM layers (`num_layers=2`), then a linear layer on the last time step. Softmax is folded into `CrossEntropyLoss`.

> Note: the original Keras code stacked two `LSTM(150)` layers without `return_sequences=True`, which errors in Keras (the 2nd LSTM would receive a 2D tensor). `nn.LSTM(num_layers=2)` is the intended stacked version.

In [18]:
class NextWordLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=150, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)          # (B, T, E)
        out, _ = self.lstm(emb)          # (B, T, H)
        return self.fc(out[:, -1, :])    # logits from last time step -> (B, V)


model = NextWordLSTM(vocab_size).to(device)
print(model)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

NextWordLSTM(
  (embedding): Embedding(395, 100, padding_idx=0)
  (lstm): LSTM(100, 150, num_layers=2, batch_first=True)
  (fc): Linear(in_features=150, out_features=395, bias=True)
)
Trainable params: 431545


## Train

In [19]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 100
for epoch in range(1, epochs + 1):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        n += xb.size(0)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{epochs} - loss: {total_loss / n:.4f} - accuracy: {correct / n:.4f}")

Epoch   1/100 - loss: 5.9590 - accuracy: 0.0292
Epoch   5/100 - loss: 5.1067 - accuracy: 0.0769
Epoch  10/100 - loss: 3.1583 - accuracy: 0.3316
Epoch  15/100 - loss: 1.8256 - accuracy: 0.6671
Epoch  20/100 - loss: 1.1053 - accuracy: 0.8501
Epoch  25/100 - loss: 0.7167 - accuracy: 0.8886
Epoch  30/100 - loss: 0.5271 - accuracy: 0.8926
Epoch  35/100 - loss: 0.4265 - accuracy: 0.8873
Epoch  40/100 - loss: 0.3639 - accuracy: 0.8833
Epoch  45/100 - loss: 0.3300 - accuracy: 0.8939
Epoch  50/100 - loss: 0.3079 - accuracy: 0.8886
Epoch  55/100 - loss: 0.2964 - accuracy: 0.8912
Epoch  60/100 - loss: 0.2835 - accuracy: 0.8886
Epoch  65/100 - loss: 0.2745 - accuracy: 0.8873
Epoch  70/100 - loss: 0.2672 - accuracy: 0.8859
Epoch  75/100 - loss: 0.2636 - accuracy: 0.8859
Epoch  80/100 - loss: 0.2546 - accuracy: 0.8886
Epoch  85/100 - loss: 0.2542 - accuracy: 0.8859
Epoch  90/100 - loss: 0.2496 - accuracy: 0.8873
Epoch  95/100 - loss: 0.2499 - accuracy: 0.8859
Epoch 100/100 - loss: 0.2461 - accuracy:

## Generate text

In [20]:
@torch.no_grad()
def predict_next_word(model, text, max_len):
    model.eval()
    token_text = tokenizer.texts_to_sequences([text])[0]
    padded = pad_sequences([token_text], max_len - 1)          # (1, 56)
    x = torch.from_numpy(padded).long().to(device)
    pos = model(x).argmax(dim=1).item()
    return tokenizer.index_word.get(pos)


text = "god bless"
for _ in range(10):
    word = predict_next_word(model, text, max_len)
    if word is None:
        break
    text = text + " " + word
    print(text)
    time.sleep(2)

god bless i
god bless i haue
god bless i haue intreated
god bless i haue intreated him
god bless i haue intreated him along
god bless i haue intreated him along along
god bless i haue intreated him along along not
god bless i haue intreated him along along not like
god bless i haue intreated him along along not like the
god bless i haue intreated him along along not like the king


In [10]:
tokenizer.word_index

{'the': 1,
 'you': 2,
 'i': 3,
 'to': 4,
 'a': 5,
 'of': 6,
 'is': 7,
 'have': 8,
 'will': 9,
 'can': 10,
 'what': 11,
 'course': 12,
 'program': 13,
 'in': 14,
 'for': 15,
 'all': 16,
 'sessions': 17,
 'on': 18,
 'be': 19,
 'and': 20,
 'this': 21,
 'if': 22,
 'am': 23,
 'pay': 24,
 'payment': 25,
 'make': 26,
 'we': 27,
 'do': 28,
 'subscription': 29,
 'where': 30,
 'rs': 31,
 'so': 32,
 'campusx': 33,
 'session': 34,
 'our': 35,
 'paid': 36,
 'join': 37,
 'able': 38,
 'your': 39,
 'website': 40,
 'placement': 41,
 'fee': 42,
 'data': 43,
 'monthly': 44,
 'month': 45,
 'not': 46,
 'get': 47,
 'yes': 48,
 'once': 49,
 'past': 50,
 'feb': 51,
 'assistance': 52,
 'science': 53,
 '7': 54,
 '5600': 55,
 'are': 56,
 'watch': 57,
 'google': 58,
 'by': 59,
 'com': 60,
 'mail': 61,
 'from': 62,
 'contact': 63,
 'us': 64,
 'at': 65,
 'or': 66,
 'doubt': 67,
 'mentorship': 68,
 'payments': 69,
 '799': 70,
 'total': 71,
 'duration': 72,
 'months': 73,
 'learning': 74,
 'case': 75,
 'here': 76,
 '